# Semantic Hybrid Recommender on Amazon Reviews

This notebook builds a staged recommendation experiment for Amazon Reviews:

```text
Popularity baseline
→ ID-based Two-Tower
→ Mini-DLRM-style ranking model
→ Hybrid Mini-DLRM + frozen semantic item embeddings
```

This project implements a compact DLRM-inspired ranking model for educational and experimental purposes, not a full-scale production DLRM system.

## Retrieval vs Ranking Clarification

This notebook evaluates models on sampled candidate sets using leave-one-out ranking. It does not implement full-scale industrial ANN retrieval over millions of items. The goal is to compare collaborative, feature-interaction, and semantic-hybrid ranking signals.

## Cold-start and Long-tail Motivation

Collaborative embeddings require user-item interactions to become meaningful. Long-tail or new items often have too few interactions to learn stable embeddings. A semantic encoder can place items into a meaningful content space based on title, category, and description before sufficient user feedback exists.

## Setup

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import torch

from src.data import (
    apply_id_mappings,
    build_category_features,
    build_dense_features,
    build_id_mappings,
    cap_interactions,
    filter_interactions,
    generate_synthetic_amazon_data,
    load_metadata,
    load_reviews,
    normalize_columns,
    temporal_split,
)
from src.metrics import evaluate_leave_one_out, metrics_to_frame
from src.models import HybridDLRM, MiniDLRM, TwoTower
from src.negative_sampling import build_eval_candidates, build_training_samples, build_user_positive_items
from src.semantic_embeddings import build_item_text, load_or_create_semantic_embeddings
from src.train import ItemFeatureStore, make_popularity_scorer, make_torch_scorer, recommend_top_k, train_model
from src.utils import get_device, seed_everything

seed_everything(42)
device = get_device()
device

In [ ]:
# Larger sparse long-tail real-data experiment configuration.
# Keep paths as None for synthetic fallback, or point them at local Amazon Reviews files.
RUN_NAME = "movies_tv_long_tail_minilm"
RUN_TIMESTAMP = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
REVIEWS_PATH = "data/raw/movies_tv_reviews_subset.jsonl"
METADATA_PATH = "data/raw/movies_tv_metadata_subset.jsonl"
MAX_INTERACTIONS = 50_000
SEMANTIC_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
EPOCHS = 5
RUN_SEMANTIC_VIS = False
SEMANTIC_VIS_METHOD = "pca"  # Use "tsne" for an optional slower visualization.

# Use SEMANTIC_MODEL_NAME = "tfidf-svd" for offline-only smoke runs.

# Model and evaluation defaults.
EMB_DIM = 32
BATCH_SIZE = 1024
LR = 1e-3
TRAIN_NEGATIVES = 4
EVAL_NEGATIVES = 99
MIN_USER_INTERACTIONS = 5
MIN_ITEM_INTERACTIONS = 2
K = 10
METRICS_OUT = REPO_ROOT / "results" / f"metrics_{RUN_NAME}_{RUN_TIMESTAMP}.csv"
LONG_TAIL_OUT = REPO_ROOT / "results" / f"long_tail_{RUN_NAME}_{RUN_TIMESTAMP}.csv"

## Load Data or Synthetic Fallback

In [ ]:
reviews_path = (REPO_ROOT / REVIEWS_PATH) if REVIEWS_PATH else None
metadata_path = (REPO_ROOT / METADATA_PATH) if METADATA_PATH else None

using_synthetic = False
if reviews_path and reviews_path.exists():
    reviews_raw = load_reviews(str(reviews_path))
    metadata_raw = load_metadata(str(metadata_path)) if metadata_path else None
else:
    using_synthetic = True
    print("Using synthetic demo dataset. For real evaluation, provide Amazon Reviews files.")
    reviews_raw, metadata_raw = generate_synthetic_amazon_data()

ACTIVE_SEMANTIC_MODEL_NAME = SEMANTIC_MODEL_NAME
if using_synthetic and SEMANTIC_MODEL_NAME != "tfidf-svd":
    print("Synthetic fallback active; using TF-IDF + SVD embeddings to keep the demo offline-safe.")
    ACTIVE_SEMANTIC_MODEL_NAME = "tfidf-svd"

reviews_raw.head(), metadata_raw.head() if metadata_raw is not None else None

## Preprocess Data

In [ ]:
reviews, metadata = normalize_columns(reviews_raw, metadata_raw)
reviews = cap_interactions(reviews, MAX_INTERACTIONS, seed=42)
interactions = filter_interactions(
    reviews,
    min_user_interactions=MIN_USER_INTERACTIONS,
    min_item_interactions=MIN_ITEM_INTERACTIONS,
)
num_stat_users = interactions["user_id"].nunique()
num_stat_items = interactions["item_id"].nunique()
num_stat_interactions = len(interactions)
sparsity = 1.0 - (num_stat_interactions / max(1, num_stat_users * num_stat_items))
dataset_stats = pd.DataFrame(
    [{"users": num_stat_users, "items": num_stat_items, "interactions": num_stat_interactions, "sparsity": sparsity}]
)
experiment_config = {
    "run_name": RUN_NAME,
    "run_timestamp": RUN_TIMESTAMP,
    "dataset": "synthetic" if using_synthetic else "Movies_and_TV",
    "users": num_stat_users,
    "items": num_stat_items,
    "interactions": num_stat_interactions,
    "sparsity": sparsity,
    "max_interactions": MAX_INTERACTIONS,
    "epochs": EPOCHS,
    "min_user_interactions": MIN_USER_INTERACTIONS,
    "min_item_interactions": MIN_ITEM_INTERACTIONS,
    "semantic_model": ACTIVE_SEMANTIC_MODEL_NAME,
    "train_negatives": TRAIN_NEGATIVES,
    "eval_negatives": EVAL_NEGATIVES,
    "k": K,
}
display(dataset_stats)
interactions.head()

## Train/Validation/Test Split

In [ ]:
train_df, val_df, test_df = temporal_split(interactions)
user_mapping, item_mapping = build_id_mappings(train_df, val_df, test_df)
train_df = apply_id_mappings(train_df, user_mapping, item_mapping)
val_df = apply_id_mappings(val_df, user_mapping, item_mapping)
test_df = apply_id_mappings(test_df, user_mapping, item_mapping)

num_users = len(user_mapping)
num_items = len(item_mapping)
print(train_df.shape, val_df.shape, test_df.shape, num_users, num_items)

## Negative Sampling

This project uses random negative sampling for simplicity. Future work can use hard negatives or popularity-aware negative sampling.

In [ ]:
all_positive_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
known_positive_items = build_user_positive_items(all_positive_df)
train_samples = build_training_samples(
    train_df,
    known_positive_items,
    num_items,
    num_negatives=TRAIN_NEGATIVES,
    seed=42,
)
val_candidates = build_eval_candidates(val_df, known_positive_items, num_items, EVAL_NEGATIVES, seed=43)
test_candidates = build_eval_candidates(test_df, known_positive_items, num_items, EVAL_NEGATIVES, seed=44)
train_samples.head(), len(val_candidates), len(test_candidates)

## Metrics

We report Recall@10, HitRate@10, and NDCG@10. For leave-one-out evaluation, Recall@K and HitRate@K are equivalent, but both are reported for readability.

## Popularity Baseline

In [ ]:
results = {}
model_scorers = {}
popularity_scorer = make_popularity_scorer(train_df, num_items)
results["Popularity"] = evaluate_leave_one_out(None, test_candidates, k=K, scorer=popularity_scorer)
results["Popularity"]

## Two-Tower Baseline

Recommendation is formulated as implicit feedback prediction. Positive samples are observed interactions and negative samples are sampled unobserved items. The model outputs raw logits, and BCEWithLogitsLoss combines sigmoid and binary cross entropy in a numerically stable way.

In [ ]:
two_tower = TwoTower(num_users, num_items, emb_dim=EMB_DIM)
two_tower = train_model(
    two_tower,
    train_samples,
    eval_candidates=val_candidates,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LR,
    k=K,
    device=device,
)
two_tower_scorer = make_torch_scorer(two_tower, device=device)
model_scorers["Two-Tower"] = two_tower_scorer
results["Two-Tower"] = evaluate_leave_one_out(
    two_tower,
    test_candidates,
    k=K,
    scorer=two_tower_scorer,
)
results["Two-Tower"]

## Mini-DLRM-style Ranking Model

Mini-DLRM learns behavioral similarity from user-item interactions and feature interactions. The model combines user, item, category, and dense item features through pairwise dot-product interactions before ranking candidates.

In [ ]:
category_by_item, category_mapping = build_category_features(item_mapping, metadata)
dense_by_item, dense_feature_names = build_dense_features(item_mapping, metadata)
item_features = ItemFeatureStore(category_by_item=category_by_item, dense_by_item=dense_by_item)
num_categories = len(category_mapping)
dense_dim = dense_by_item.shape[1]
num_categories, dense_dim, dense_feature_names

In [ ]:
mini_dlrm = MiniDLRM(num_users, num_items, num_categories, dense_dim, emb_dim=EMB_DIM)
mini_dlrm = train_model(
    mini_dlrm,
    train_samples,
    eval_candidates=val_candidates,
    item_features=item_features,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LR,
    k=K,
    device=device,
)
mini_dlrm_scorer = make_torch_scorer(mini_dlrm, item_features=item_features, device=device)
model_scorers["Mini-DLRM"] = mini_dlrm_scorer
results["Mini-DLRM"] = evaluate_leave_one_out(
    mini_dlrm,
    test_candidates,
    k=K,
    scorer=mini_dlrm_scorer,
)
results["Mini-DLRM"]

## Semantic Item Embeddings

The goal is not to fine-tune a text embedding model. Instead, pretrained encoders such as MiniLM, BGE, or e5 are used as frozen semantic feature extractors. This is faster, avoids overfitting on a small recommendation dataset, and preserves pretrained semantic structure.

Embedding generation can be expensive, so semantic vectors are cached and reused across runs.

In [ ]:
item_texts = build_item_text(metadata, item_mapping)
semantic_embeddings = load_or_create_semantic_embeddings(
    item_texts,
    cache_path=REPO_ROOT / "data" / "embeddings" / "semantic_item_embeddings.npy",
    model_name=ACTIVE_SEMANTIC_MODEL_NAME,
)
semantic_dim = semantic_embeddings.shape[1]
semantic_embeddings.shape

## Hybrid Mini-DLRM + Semantic Embeddings

Semantic embeddings may be 384-dimensional, while DLRM-style feature vectors use 32 dimensions. A trainable projection layer maps semantic vectors into the recommender latent space so they can participate in feature interactions.

DLRM-style models learn behavioral similarity from interactions. Semantic encoders learn content similarity from product text. The hybrid model combines both signals, which is especially useful for sparse, long-tail, and cold-start items.

In [ ]:
hybrid = HybridDLRM(
    num_users,
    num_items,
    num_categories,
    dense_dim,
    semantic_embeddings=semantic_embeddings,
    semantic_dim=semantic_dim,
    emb_dim=EMB_DIM,
)
hybrid = train_model(
    hybrid,
    train_samples,
    eval_candidates=val_candidates,
    item_features=item_features,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LR,
    k=K,
    device=device,
)
hybrid_scorer = make_torch_scorer(hybrid, item_features=item_features, device=device)
model_scorers["Hybrid Mini-DLRM + Semantic"] = hybrid_scorer
results["Hybrid Mini-DLRM + Semantic"] = evaluate_leave_one_out(
    hybrid,
    test_candidates,
    k=K,
    scorer=hybrid_scorer,
)
results["Hybrid Mini-DLRM + Semantic"]

## Evaluation Table

In [ ]:
metrics_df = metrics_to_frame(results)
for key, value in experiment_config.items():
    metrics_df[key] = value
METRICS_OUT.parent.mkdir(parents=True, exist_ok=True)
metrics_df.to_csv(METRICS_OUT, index=False)
print(f"Saved metrics to {METRICS_OUT}")
metrics_df[["Model", f"Recall@{K}", f"HitRate@{K}", f"NDCG@{K}", "items", "interactions", "epochs", "semantic_model"]]

## Ablation Summary

In [ ]:
pd.DataFrame(
    [
        {"Model": "Popularity", "Collaborative Signal": "No", "Feature Interaction": "No", "Semantic Signal": "No"},
        {"Model": "Two-Tower", "Collaborative Signal": "Yes", "Feature Interaction": "No", "Semantic Signal": "No"},
        {"Model": "Mini-DLRM", "Collaborative Signal": "Yes", "Feature Interaction": "Yes", "Semantic Signal": "No"},
        {"Model": "Hybrid Mini-DLRM + Semantic", "Collaborative Signal": "Yes", "Feature Interaction": "Yes", "Semantic Signal": "Yes"},
    ]
)

## Long-tail Analysis

Head items are the top 20% by train popularity. Tail items are the bottom 80%. The table below evaluates Two-Tower, Mini-DLRM, and Hybrid Mini-DLRM + Semantic separately on head and tail test positives. The hypothesis is that the hybrid semantic model should help more on tail items because it can use item content semantics when collaborative interactions are sparse.

In [ ]:
def filter_candidates_by_items(candidates, allowed_items):
    allowed_items = set(allowed_items)
    return [row for row in candidates if row["true_item"] in allowed_items]

popularity = train_df["item_idx"].value_counts()
head_cutoff = max(1, int(np.ceil(num_items * 0.2)))
head_items = set(popularity.sort_values(ascending=False).head(head_cutoff).index.astype(int))
tail_items = set(range(num_items)) - head_items

long_tail_rows = []
for model_name, scorer in model_scorers.items():
    for group_name, item_group in [("head", head_items), ("tail", tail_items)]:
        group_candidates = filter_candidates_by_items(test_candidates, item_group)
        group_metrics = evaluate_leave_one_out(None, group_candidates, k=K, scorer=scorer)
        long_tail_rows.append(
            {
                "Model": model_name,
                "Group": group_name,
                "Test Users": len(group_candidates),
                "Test Items": len({row["true_item"] for row in group_candidates}),
                **group_metrics,
            }
        )

long_tail_df = pd.DataFrame(long_tail_rows)
for key, value in experiment_config.items():
    long_tail_df[key] = value
LONG_TAIL_OUT.parent.mkdir(parents=True, exist_ok=True)
long_tail_df.to_csv(LONG_TAIL_OUT, index=False)
print(f"Saved long-tail metrics to {LONG_TAIL_OUT}")
long_tail_df[["Model", "Group", "Test Users", "Test Items", f"Recall@{K}", f"HitRate@{K}", f"NDCG@{K}"]]

## Example Recommendation Demo

This small product-style demo shows one user's history and top Hybrid recommendations with item metadata snippets.

In [ ]:
idx_to_item = {idx: item_id for item_id, idx in item_mapping.items()}
metadata_lookup = metadata.drop_duplicates("item_id").set_index("item_id") if metadata is not None else pd.DataFrame()

demo_user = int(test_df.iloc[0]["user_idx"])
known_items = known_positive_items.get(demo_user, set())
candidate_items = list(range(num_items))
recommended = recommend_top_k(hybrid_scorer, demo_user, candidate_items, known_items=known_items, k=5)

def describe_item(item_idx):
    item_id = idx_to_item[item_idx]
    if item_id in metadata_lookup.index:
        row = metadata_lookup.loc[item_id]
        return {
            "item_id": item_id,
            "title": row.get("title", ""),
            "category": row.get("main_category", ""),
            "description": str(row.get("description", ""))[:120],
        }
    return {"item_id": item_id, "title": item_id, "category": "", "description": ""}

history = [describe_item(item_idx) for item_idx in list(known_items)[:5]]
recs = [describe_item(item_idx) for item_idx in recommended]
print(f"Demo user_idx: {demo_user}")
print("History sample")
display(pd.DataFrame(history))
print("Hybrid recommendations")
display(pd.DataFrame(recs))

## Optional Semantic Embedding Visualization

If desired, run PCA or t-SNE on `semantic_embeddings` and color by category to inspect whether item content clusters by category. This is intentionally optional and not required for tests.

In [ ]:
if RUN_SEMANTIC_VIS:
    import matplotlib.pyplot as plt

    if SEMANTIC_VIS_METHOD.lower() == "tsne":
        from sklearn.manifold import TSNE

        perplexity = min(30, max(5, (len(semantic_embeddings) - 1) // 3))
        coords = TSNE(n_components=2, perplexity=perplexity, init="pca", learning_rate="auto", random_state=42).fit_transform(
            semantic_embeddings
        )
    else:
        from sklearn.decomposition import PCA

        coords = PCA(n_components=2, random_state=42).fit_transform(semantic_embeddings)

    idx_to_category = {idx: category for category, idx in category_mapping.items()}
    plot_df = pd.DataFrame(
        {
            "x": coords[:, 0],
            "y": coords[:, 1],
            "category": [idx_to_category.get(int(category_idx), "unknown") for category_idx in category_by_item],
        }
    )
    if len(plot_df) > 2_000:
        plot_df = plot_df.sample(2_000, random_state=42)

    plt.figure(figsize=(8, 6))
    top_categories = plot_df["category"].value_counts().head(12).index
    for category, group in plot_df.groupby(plot_df["category"].where(plot_df["category"].isin(top_categories), "other")):
        plt.scatter(group["x"], group["y"], s=14, alpha=0.7, label=str(category)[:35])
    plt.title(f"Semantic item embedding visualization ({SEMANTIC_VIS_METHOD.upper()})")
    plt.xlabel("component 1")
    plt.ylabel("component 2")
    plt.legend(loc="best", fontsize=8)
    plt.show()
else:
    print("Semantic visualization skipped. Set RUN_SEMANTIC_VIS = True to enable PCA/t-SNE.")

## Real-data Analysis Notes

The previous tiny real-data pass used a very small item universe, which was too dense and repetitive for semantic item representations to show much value. In that regime, Two-Tower and Mini-DLRM can look similar because sampled candidate ranking is dominated by repeated collaborative signals, and the DLRM-style feature interaction layer does not have enough item diversity to learn a meaningfully different decision boundary.

This larger sparse long-tail setting is the more relevant test for the project hypothesis. Semantic embeddings are expected to matter more when item diversity is larger, item interactions are sparse, and tail items still have useful title/category/description text. Compare the Two-Tower, Mini-DLRM, and Hybrid Mini-DLRM + Semantic rows in the head/tail table. If the hybrid model improves tail Recall@10 or NDCG@10, that suggests pretrained product-text semantics are adding useful item structure beyond collaborative IDs. If it underperforms, inspect metadata coverage, semantic cache quality, training epochs, and whether random negatives are too easy before drawing a final conclusion.

## Final Takeaways

The staged experiment separates popularity, collaborative ID embeddings, feature interaction, and semantic content signals. The hybrid ranker is designed to retain collaborative behavior while giving long-tail and cold-start items a content-based representation before they accumulate many interactions.

## Future Work

- hard negative mining
- popularity-aware negative sampling
- ANN retrieval with Faiss
- SASRec or BERT4Rec sequence modeling
- LLM reranking
- multimodal product embeddings
- fine-tuning semantic encoders on recommendation pairs